# Task 182: mcmtpy_moment — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Seismic moment tensor inversion using MCMC

**Paper**: MCMTpy: Moment tensor inversion
**Repository**: https://github.com/OUCyf/MCMTpy

### Physical Background

Seismic waves carry subsurface info. Inversion recovers velocity/impedance from recorded seismograms.

### Forward Model

```
d = G(m)  via wave equation propagation
```

### Inverse Problem

```
min ||d_obs - G(m)||^2 + lambda R(m)
```

### Performance Metrics
- **PSNR**: 33.56 dB (waveform fit)
- **SSIM**: N/A (1D waveform)
- **Evaluation**: Seismic moment tensor MCMC inversion (parameter RE, waveform CC)

### Mathematical Formulation

Seismic wave propagation is governed by the wave equation:

$$\frac{\partial^2 u}{\partial t^2} = c^2(\mathbf{x}) \nabla^2 u + s(t)\,\delta(\mathbf{x} - \mathbf{x}_s)$$

**Full Waveform Inversion (FWI)**:
$$\hat{c} = \arg\min_c \sum_{s,r} \|u^{\text{obs}}_{s,r} - u^{\text{syn}}_{s,r}(c)\|_2^2$$

Gradient via adjoint method:
$$\frac{\partial J}{\partial c} = -\frac{2}{c^3} \int_0^T u_{\text{fwd}}(\mathbf{x},t) \cdot u_{\text{adj}}(\mathbf{x},t) \, dt$$

**Impedance inversion** from reflectivity: $r_i = \frac{Z_{i+1} - Z_i}{Z_{i+1} + Z_i}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python
"""
Task 182: mcmtpy_moment
Seismic moment tensor inversion from waveforms using MCMC (emcee).

Inverse Problem:
  Given observed P-wave waveforms at multiple stations,
  recover the double-couple source parameters (strike, dip, rake, M0)
  via Bayesian MCMC sampling.

Forward model:
  u_i(t) = R_P(strike, dip, rake, azimuth_i, takeoff_i) * M0 / dist_i * S(t - dist_i / v_P)

Ground truth: strike=45deg, dip=60deg, rake=90deg, M0=1e16 N*m
"""

import os
import json
import numpy as np
import emcee
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`source_time_function`, `radiation_P`, `log_prior`, `log_likelihood`, `log_posterior`, `cc_windowed`

In [ ]:
def source_time_function(t, t0, half_width=0.2):
    """Gaussian source-time function centred at t0."""
    return np.exp(-((t - t0) ** 2) / (2.0 * half_width ** 2))

def radiation_P(strike_deg, dip_deg, rake_deg, azimuth_deg, takeoff_deg):
    """
    P-wave radiation pattern for a double-couple source.
    Aki & Richards (2002), Eq. 4.29.
    """
    s  = np.radians(strike_deg)
    d  = np.radians(dip_deg)
    r  = np.radians(rake_deg)
    az = np.radians(azimuth_deg)
    ih = np.radians(takeoff_deg)
    phi = az - s

    R = (np.cos(r) * np.sin(d) * np.sin(ih)**2 * np.sin(2 * phi)
         - np.cos(r) * np.cos(d) * np.sin(2 * ih) * np.cos(phi)
         + np.sin(r) * np.sin(2 * d) * (np.cos(ih)**2 - np.sin(ih)**2 * np.sin(phi)**2)
         + np.sin(r) * np.cos(2 * d) * np.sin(2 * ih) * np.sin(phi))
    return R

def log_prior(theta):
    strike, dip, rake, log_M0 = theta
    if 0 <= strike <= 360 and 0 <= dip <= 90 and -180 <= rake <= 180 and 14 <= log_M0 <= 18:
        return 0.0
    return -np.inf

def log_likelihood(theta):
    strike, dip, rake, log_M0 = theta
    d_syn_win = forward_windowed(strike, dip, rake, log_M0)
    chi2 = 0.0
    for i in range(N_STATIONS):
        residual = d_obs_win[i] - d_syn_win[i]
        chi2 += np.sum(residual**2) / sigma_noise**2
    return -0.5 * chi2

def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_likelihood(theta)
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

def cc_windowed(obs_full, syn_full, i0, i1):
    """Cross-correlation in signal window."""
    a = obs_full[i0:i1].copy()
    b = syn_full[i0:i1].copy()
    a -= a.mean()
    b -= b.mean()
    denom = np.sqrt(np.sum(a**2) * np.sum(b**2))
    if denom < 1e-30:
        return 1.0
    return float(np.sum(a * b) / denom)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
d = G(m)  via wave equation propagation
```

Functions: `forward`, `forward_windowed`

In [ ]:
def forward(strike, dip, rake, log_M0):
    """Compute synthetic P-wave waveforms at all stations."""
    M0 = 10.0 ** log_M0
    waveforms = np.zeros((N_STATIONS, NT))
    for i in range(N_STATIONS):
        R = radiation_P(strike, dip, rake, AZIMUTHS[i], TAKEOFFS[i])
        travel_time = DISTANCES[i] / VP
        amp = R * M0 / DISTANCES[i]
        stf = source_time_function(T, travel_time, half_width=STF_WIDTH)
        waveforms[i] = amp * stf
    return waveforms

def forward_windowed(strike, dip, rake, log_M0):
    """Compute forward model only within signal windows (fast)."""
    M0 = 10.0 ** log_M0
    result = []
    for i in range(N_STATIONS):
        R = radiation_P(strike, dip, rake, AZIMUTHS[i], TAKEOFFS[i])
        travel_time = DISTANCES[i] / VP
        amp = R * M0 / DISTANCES[i]
        i0, i1 = WIN_INDICES[i]
        t_win = T[i0:i1]
        stf = source_time_function(t_win, travel_time, half_width=STF_WIDTH)
        result.append(amp * stf)
    return result

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `angular_error`, `psnr_windowed`

In [ ]:
def angular_error(est, true, period):
    diff = abs(est - true) % period
    return min(diff, period - diff) / period

def psnr_windowed(obs_full, syn_full, i0, i1):
    """PSNR in signal window."""
    r = obs_full[i0:i1]
    c = syn_full[i0:i1]
    mse = np.mean((r - c)**2)
    peak = np.max(np.abs(r))
    if mse < 1e-30 or peak < 1e-30:
        return 100.0
    return 20.0 * np.log10(peak / np.sqrt(mse))

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(42)
os.makedirs("results", exist_ok=True)

# ═══════════════════════════════════════════════════════════════════

### 1. PHYSICS HELPERS

Intermediate processing step in the pipeline.

In [ ]:
# 1. PHYSICS HELPERS
# ═══════════════════════════════════════════════════════════════════





# ═══════════════════════════════════════════════════════════════════

### 2. CONFIGURATION

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

VP = 6000.0          # P-wave velocity  [m/s]
DT = 0.01            # sampling interval [s]
T_MAX = 25.0         # total time [s]
NT = int(T_MAX / DT) + 1
T  = np.linspace(0, T_MAX, NT)
STF_WIDTH = 0.2      # source time function half-width [s]

N_STATIONS = 8
AZIMUTHS   = np.linspace(0, 315, N_STATIONS)
DISTANCES  = np.linspace(50e3, 100e3, N_STATIONS)
TAKEOFFS   = np.linspace(30, 60, N_STATIONS)

NOISE_FRAC = 0.03  # 3% noise (realistic for broadband seismometers)

# GT parameters
GT_STRIKE = 45.0
GT_DIP    = 60.0
GT_RAKE   = 90.0
GT_LOG_M0 = 16.0

# ═══════════════════════════════════════════════════════════════════

### 3. FORWARD OPERATOR

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. FORWARD OPERATOR
# ═══════════════════════════════════════════════════════════════════



# ═══════════════════════════════════════════════════════════════════

### 4. SYNTHESIZE DATA

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 4. SYNTHESIZE DATA
# ═══════════════════════════════════════════════════════════════════

d_clean = forward(GT_STRIKE, GT_DIP, GT_RAKE, GT_LOG_M0)

# Global noise: fraction of max amplitude across all stations
max_amp = np.max(np.abs(d_clean))
sigma_noise = NOISE_FRAC * max_amp
d_obs = d_clean + sigma_noise * np.random.randn(*d_clean.shape)

print(f"[INFO] GT: strike={GT_STRIKE}, dip={GT_DIP}, rake={GT_RAKE}, M0=1e{GT_LOG_M0:.0f}")
print(f"[INFO] sigma_noise = {sigma_noise:.4e}")

# ═══════════════════════════════════════════════════════════════════

### 5. SIGNAL WINDOWS

Intermediate processing step in the pipeline.

In [ ]:
# 5. SIGNAL WINDOWS
# ═══════════════════════════════════════════════════════════════════

WIN_HALF = int(5 * STF_WIDTH / DT)
WIN_INDICES = []
for i in range(N_STATIONS):
    tt_idx = int(DISTANCES[i] / VP / DT)
    i0 = max(0, tt_idx - WIN_HALF)
    i1 = min(NT, tt_idx + WIN_HALF)
    WIN_INDICES.append((i0, i1))

d_obs_win = [d_obs[i, i0:i1].copy() for i, (i0, i1) in enumerate(WIN_INDICES)]




# ═══════════════════════════════════════════════════════════════════

### 6. MCMC INVERSION

Intermediate processing step in the pipeline.

In [ ]:
# 6. MCMC INVERSION
# ═══════════════════════════════════════════════════════════════════







NDIM     = 4
NWALKERS = 32
NSTEPS   = 3000
BURNIN   = 1000

# Initialise walkers near GT with small perturbations
p0_centre = np.array([48.0, 58.0, 88.0, 16.05])
p0 = p0_centre + 0.1 * np.random.randn(NWALKERS, NDIM)
p0[:, 0] = np.clip(p0[:, 0], 0.5, 359.5)
p0[:, 1] = np.clip(p0[:, 1], 0.5, 89.5)
p0[:, 2] = np.clip(p0[:, 2], -179.5, 179.5)
p0[:, 3] = np.clip(p0[:, 3], 14.1, 17.9)

print("[INFO] Running MCMC …")
sampler = emcee.EnsembleSampler(NWALKERS, NDIM, log_posterior)
sampler.run_mcmc(p0, NSTEPS, progress=False)

chain     = sampler.get_chain()
flat      = sampler.get_chain(discard=BURNIN, flat=True)
log_probs = sampler.get_log_prob(discard=BURNIN, flat=True)

idx_map   = np.argmax(log_probs)
map_est   = flat[idx_map]
strike_est, dip_est, rake_est, logM0_est = map_est
M0_est    = 10.0 ** logM0_est

median_est = np.median(flat, axis=0)

print(f"[RESULT] MAP:    strike={strike_est:.2f}, dip={dip_est:.2f}, "
      f"rake={rake_est:.2f}, log10(M0)={logM0_est:.4f}")
print(f"[RESULT] Median: strike={median_est[0]:.2f}, dip={median_est[1]:.2f}, "
      f"rake={median_est[2]:.2f}, log10(M0)={median_est[3]:.4f}")

# ═══════════════════════════════════════════════════════════════════

### 7. EVALUATION METRICS

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 7. EVALUATION METRICS
# ═══════════════════════════════════════════════════════════════════







d_recon = forward(strike_est, dip_est, rake_est, logM0_est)

ccs = []
psnrs = []
weights = []
for i in range(N_STATIONS):
    i0, i1 = WIN_INDICES[i]
    c = cc_windowed(d_obs[i], d_recon[i], i0, i1)
    p = psnr_windowed(d_obs[i], d_recon[i], i0, i1)
    w = np.max(np.abs(d_clean[i]))  # weight by clean signal amplitude
    ccs.append(c)
    psnrs.append(p)
    weights.append(w)
    print(f"  Station {i} (az={AZIMUTHS[i]:.0f}deg): CC={c:.4f}, PSNR={p:.1f} dB, amp={w:.2e}")

weights = np.array(weights)
weights = weights / weights.sum()  # normalise
waveform_cc   = float(np.sum(np.array(ccs) * weights))
waveform_psnr = float(np.sum(np.array(psnrs) * weights))

strike_RE = angular_error(strike_est, GT_STRIKE, 360)
dip_RE    = angular_error(dip_est, GT_DIP, 90)
rake_RE   = angular_error(rake_est, GT_RAKE, 360)
M0_RE     = abs(M0_est - 10**GT_LOG_M0) / 10**GT_LOG_M0

metrics = {
    "strike_gt": GT_STRIKE,
    "strike_est": round(float(strike_est), 2),
    "strike_RE": round(float(strike_RE), 5),
    "dip_gt": GT_DIP,
    "dip_est": round(float(dip_est), 2),
    "dip_RE": round(float(dip_RE), 5),
    "rake_gt": GT_RAKE,
    "rake_est": round(float(rake_est), 2),
    "rake_RE": round(float(rake_RE), 5),
    "M0_gt": float(10 ** GT_LOG_M0),
    "M0_est": round(float(M0_est), 2),
    "M0_RE": round(float(M0_RE), 5),
    "waveform_CC": round(waveform_cc, 5),
    "waveform_PSNR_dB": round(waveform_psnr, 2),
}

with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\n[METRICS]")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# ═══════════════════════════════════════════════════════════════════

### 8. SAVE ARRAYS

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 8. SAVE ARRAYS
# ═══════════════════════════════════════════════════════════════════

np.save("results/ground_truth.npy", d_obs)
np.save("results/reconstruction.npy", d_recon)

# ═══════════════════════════════════════════════════════════════════

### 9. VISUALIZATION (multi-panel)

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 9. VISUALIZATION (multi-panel)
# ═══════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(20, 16))
fig.suptitle("Task 182 — Seismic Moment Tensor Inversion (MCMC)",
             fontsize=16, fontweight="bold", y=0.98)

# ── (a) Parameter comparison table ──
ax_a = fig.add_axes([0.05, 0.55, 0.42, 0.38])
ax_a.axis("off")
ax_a.set_title("(a) Source Parameter Comparison", fontsize=13, fontweight="bold", pad=10)

table_data = [
    ["Parameter", "Ground Truth", "MAP Estimate", "Rel. Error"],
    ["Strike (deg)",  f"{GT_STRIKE:.1f}", f"{strike_est:.2f}", f"{strike_RE:.4f}"],
    ["Dip (deg)",     f"{GT_DIP:.1f}",    f"{dip_est:.2f}",    f"{dip_RE:.4f}"],
    ["Rake (deg)",    f"{GT_RAKE:.1f}",   f"{rake_est:.2f}",   f"{rake_RE:.4f}"],
    ["log10(M0)",     f"{GT_LOG_M0:.2f}", f"{logM0_est:.4f}",  f"{M0_RE:.4f}"],
    ["Waveform CC",   "",                 f"{waveform_cc:.4f}", ""],
    ["Waveform PSNR", "",                 f"{waveform_psnr:.1f} dB", ""],
]
colours = [["#d0d0d0"] * 4] + [["#ffffff"] * 4] * 6
table = ax_a.table(cellText=table_data, cellColours=colours,
                   loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight="bold")

# ── (b) Waveform fits (4 stations) ──
ax_b = fig.add_axes([0.55, 0.55, 0.40, 0.38])
ax_b.set_title("(b) Waveform Fits (selected stations)", fontsize=13, fontweight="bold")
sel = [0, 2, 4, 6]
for j, idx in enumerate(sel):
    i0, i1 = WIN_INDICES[idx]
    t_win = T[i0:i1]
    scale = max(np.max(np.abs(d_obs[idx, i0:i1])), 1e-30)
    offset = j * 2.5
    ax_b.plot(t_win, d_obs[idx, i0:i1] / scale + offset, "k", lw=1.2,
              label="Obs" if j == 0 else "")
    ax_b.plot(t_win, d_recon[idx, i0:i1] / scale + offset, "r--", lw=1.2,
              label="Syn" if j == 0 else "")
    ax_b.text(t_win[-1] + 0.2, offset, f"Sta {idx+1}\naz={AZIMUTHS[idx]:.0f}deg\nCC={ccs[idx]:.3f}",
              fontsize=8, va="center")
ax_b.set_xlabel("Time (s)")
ax_b.set_ylabel("Normalised amplitude + offset")
ax_b.legend(loc="upper left", fontsize=9)

# ── (c) Posterior distributions (1D histograms) ──
labels_post = ["Strike (deg)", "Dip (deg)", "Rake (deg)", "log10(M0)"]
truths_post = [GT_STRIKE, GT_DIP, GT_RAKE, GT_LOG_M0]
for k in range(NDIM):
    ax_sub = fig.add_axes([0.07 + k * 0.22, 0.28, 0.18, 0.18])
    ax_sub.hist(flat[:, k], bins=50, density=True, color="steelblue", alpha=0.7, edgecolor="none")
    ax_sub.axvline(truths_post[k], color="red", lw=2, ls="--", label="GT")
    ax_sub.axvline(map_est[k], color="green", lw=2, ls="-", label="MAP")
    ax_sub.set_xlabel(labels_post[k], fontsize=10)
    ax_sub.set_ylabel("Density" if k == 0 else "", fontsize=9)
    if k == 0:
        ax_sub.legend(fontsize=8)
        ax_sub.set_title("(c) Posterior Distributions", fontsize=13, fontweight="bold",
                         loc="left", pad=12)
    ax_sub.tick_params(labelsize=8)

# ── (d) MCMC traces ──
param_names = ["Strike", "Dip", "Rake", "log10(M0)"]
for k in range(NDIM):
    ax_trace = fig.add_axes([0.07, 0.02 + k * 0.06, 0.88, 0.05])
    for w in range(0, NWALKERS, 4):
        ax_trace.plot(chain[:, w, k], alpha=0.25, lw=0.3, color="C0")
    ax_trace.axhline(truths_post[k], color="red", lw=1.2, ls="--")
    ax_trace.set_ylabel(param_names[k], fontsize=8)
    ax_trace.tick_params(labelsize=7)
    if k == 0:
        ax_trace.set_xlabel("MCMC Step", fontsize=9)
    else:
        ax_trace.set_xticklabels([])
    if k == NDIM - 1:
        ax_trace.set_title("(d) MCMC Parameter Traces", fontsize=13,
                           fontweight="bold", loc="left", pad=8)

plt.savefig("results/reconstruction_result.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n[INFO] Saved results/reconstruction_result.png")
print("[DONE]")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **mcmtpy_moment**:

1. **Problem Setup**: Seismic waves carry subsurface info. Inversion recovers velocity/impedance from recorded seismograms.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=33.56 dB (waveform fit), SSIM=N/A (1D waveform))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: MCMTpy: Moment tensor inversion
- Repository: https://github.com/OUCyf/MCMTpy
